# EDA датасета CoAT

Интерактивная замена `eda.py`. Ноутбук загружает конфигурацию `binary`, не меняет исходные данные и сохраняет результаты в `reports/eda/`.


In [ ]:
from pathlib import Path
import json, hashlib
import matplotlib.pyplot as plt
import pandas as pd
from datasets import load_dataset

REPORTS = Path('reports') if Path('reports').exists() else Path('../reports')
OUT = REPORTS / 'eda'; OUT.mkdir(parents=True, exist_ok=True)
dataset = load_dataset('RussianNLP/coat', 'binary')
dataset


In [ ]:
frames = {name: split.to_pandas() for name, split in dataset.items()}
schema = {'dataset': 'RussianNLP/coat', 'configuration': 'binary',
          'splits': {name: len(frame) for name, frame in frames.items()},
          'columns': frames['train'].columns.tolist(),
          'features': {name: str(feature) for name, feature in dataset['train'].features.items()}}
(OUT / 'schema.json').write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')
pd.DataFrame([{'split': name, 'examples': len(frame), 'columns': ', '.join(frame.columns)} for name, frame in frames.items()])


In [ ]:
rows = []
for name, frame in frames.items():
    text = frame['text'].fillna('')
    words = text.str.findall(r'\S+').str.len()
    for unit, values in [('characters', text.str.len()), ('words', words)]:
        rows.append({'split': name, 'unit': unit, 'min': int(values.min()), 'max': int(values.max()), 'mean': float(values.mean()), 'median': float(values.median())})
lengths = pd.DataFrame(rows); lengths.to_csv(OUT / 'length_statistics.csv', index=False, encoding='utf-8')
classes = pd.concat([frame.assign(split=name).groupby(['split','label']).size().rename('count').reset_index() for name, frame in frames.items()])
classes.to_csv(OUT / 'class_distribution.csv', index=False, encoding='utf-8')
lengths


In [ ]:
quality, hashes = [], {}
for name, frame in frames.items():
    text = frame['text'].fillna('')
    digest = text.map(lambda value: hashlib.sha256(value.encode('utf-8')).hexdigest())
    hashes[name] = set(digest)
    quality.append({'split': name, 'empty_or_whitespace_texts': int((text.str.strip() == '').sum()), 'duplicate_rows_within_split': int(len(frame) - text.nunique())})
quality = pd.DataFrame(quality); quality.to_csv(OUT / 'quality_checks.csv', index=False, encoding='utf-8')
overlaps = []
names = list(hashes)
for i, left in enumerate(names):
    for right in names[i + 1:]: overlaps.append({'split_a': left, 'split_b': right, 'shared_unique_exact_texts': len(hashes[left] & hashes[right])})
pd.DataFrame(overlaps).to_csv(OUT / 'split_overlaps.csv', index=False, encoding='utf-8')
quality, pd.DataFrame(overlaps)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
classes.pivot(index='split', columns='label', values='count').plot(kind='bar', ax=axes[0], ylabel='Количество текстов', title='Распределение классов')
for name, frame in frames.items(): axes[1].hist(frame['text'].fillna('').str.findall(r'\S+').str.len(), bins=60, alpha=.45, label=name)
axes[1].set(xlabel='Число слов', ylabel='Количество текстов', title='Распределение длины'); axes[1].legend(); fig.tight_layout()
fig.savefig(OUT / 'length_distributions.png', dpi=300)
plt.show()
